In [ ]:
# 🏀 March Madness 2025 - Ultimate Model with Feature Engineering, Stacking & Bayesian Optimization
import pandas as pd
import numpy as np
import os
import time
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from skopt import BayesSearchCV  # Bayesian Optimization
from skopt.space import Real, Integer

# 📂 Define the data path
data_path = "data/"

# 🏀 Load Data
mteams = pd.read_csv(os.path.join(data_path, "MTeams.csv"))
wteams = pd.read_csv(os.path.join(data_path, "WTeams.csv"))
mregular_results = pd.read_csv(os.path.join(data_path, "MRegularSeasonDetailedResults.csv"))  # Use detailed stats
wregular_results = pd.read_csv(os.path.join(data_path, "WRegularSeasonDetailedResults.csv"))
mtourney_results = pd.read_csv(os.path.join(data_path, "MNCAATourneyCompactResults.csv"))
wtourney_results = pd.read_csv(os.path.join(data_path, "WNCAATourneyCompactResults.csv"))
mtourney_seeds = pd.read_csv(os.path.join(data_path, "MNCAATourneySeeds.csv"))
wtourney_seeds = pd.read_csv(os.path.join(data_path, "WNCAATourneySeeds.csv"))

# ✅ Extract numeric seed values
def extract_seed(seed):
    return int(seed[1:3])

mtourney_seeds["Seed"] = mtourney_seeds["Seed"].apply(extract_seed)
wtourney_seeds["Seed"] = wtourney_seeds["Seed"].apply(extract_seed)

# ✅ Compute Advanced Team Statistics
def compute_team_stats(results):
    team_stats = results.groupby(["Season", "WTeamID"]).agg({
        "WScore": ["mean", "sum"],
        "LScore": ["mean", "sum"],
        "NumOT": "sum",
        "WFGM": "sum",
        "WFGA": "sum",
        "WFGM3": "sum",
        "WFGA3": "sum",
        "WFTM": "sum",
        "WFTA": "sum",
        "WOR": "sum",
        "WDR": "sum",
        "WAst": "sum",
        "WTO": "sum",
        "WStl": "sum",
        "WBlk": "sum",
    }).reset_index()

    team_stats.columns = [
        "Season", "TeamID", "AvgWScore", "TotalWScore", "AvgLScore", "TotalLScore",
        "OTGames", "FGMade", "FGAttempted", "ThreePtMade", "ThreePtAttempted",
        "FTMade", "FTAttempted", "OffRebounds", "DefRebounds",
        "Assists", "Turnovers", "Steals", "Blocks"
    ]

    # 🔹 Feature Engineering
    team_stats["ScoreMargin"] = team_stats["TotalWScore"] - team_stats["TotalLScore"]
    team_stats["WinRate"] = team_stats["TotalWScore"] / (team_stats["TotalWScore"] + team_stats["TotalLScore"])
    team_stats["TurnoverMargin"] = team_stats["Turnovers"] - team_stats["Assists"]
    team_stats["AssistRatio"] = team_stats["Assists"] / (team_stats["FGAttempted"] + team_stats["FTAttempted"])
    team_stats["ThreePointRate"] = team_stats["ThreePtMade"] / team_stats["FGAttempted"]
    team_stats["FreeThrowRate"] = team_stats["FTMade"] / team_stats["FTAttempted"]
    team_stats["ReboundMargin"] = team_stats["OffRebounds"] - team_stats["DefRebounds"]
    team_stats["StealRate"] = team_stats["Steals"] / (team_stats["FGAttempted"] + team_stats["FTAttempted"])
    team_stats["BlockRate"] = team_stats["Blocks"] / (team_stats["FGAttempted"])

    return team_stats

mteam_stats = compute_team_stats(mregular_results)
wteam_stats = compute_team_stats(wregular_results)

# ✅ Prepare Training Data
def prepare_training_data(results, seeds, team_stats):
    results = results.merge(seeds, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "WSeed"}).drop(columns=["TeamID"])
    results = results.merge(seeds, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "LSeed"}).drop(columns=["TeamID"])

    results = results.merge(team_stats.rename(columns={"TeamID": "WTeamID"}), on=["Season", "WTeamID"], how="left")
    results = results.merge(team_stats.rename(columns={"TeamID": "LTeamID"}), on=["Season", "LTeamID"], how="left", suffixes=("_W", "_L"))

    results.fillna(0, inplace=True)

    # 🔹 Feature Selection
    results["SeedDiff"] = results["WSeed"] - results["LSeed"]
    results["WinRateDiff"] = results["WinRate_W"] - results["WinRate_L"]
    results["ScoreMarginDiff"] = results["ScoreMargin_W"] - results["ScoreMargin_L"]

    feature_columns = [
        "SeedDiff", "WinRateDiff", "ScoreMarginDiff"
    ]

    win_features = results[feature_columns].copy()
    win_features["Win"] = 1  

    loss_features = results[feature_columns].copy()
    loss_features["Win"] = 0  

    return pd.concat([win_features, loss_features], ignore_index=True)

mtrain_data = prepare_training_data(mtourney_results, mtourney_seeds, mteam_stats)
wtrain_data = prepare_training_data(wtourney_results, wtourney_seeds, wteam_stats)
full_train_data = pd.concat([mtrain_data, wtrain_data], ignore_index=True)

# 🏀 Split and Scale Data
X = full_train_data.drop(columns=["Win"])
y = full_train_data["Win"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# 🏀 Stacking Classifier
base_models = [
    ("rf", RandomForestClassifier(n_estimators=200)),
    ("gb", GradientBoostingClassifier(n_estimators=200)),
    ("xgb", XGBClassifier(n_estimators=200))
]
stack_model = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression())
stack_model.fit(X_train_scaled, y_train)
stack_log_loss = log_loss(y_valid, stack_model.predict_proba(X_valid_scaled)[:, 1])
print(f"🔹 Stacked Model Log Loss: {stack_log_loss}")

# 🏀 Bayesian Hyperparameter Optimization
xgb = XGBClassifier()

param_space = {
    "max_depth": Integer(3, 10),
    "learning_rate": Real(0.01, 0.3, "log-uniform"),
    "n_estimators": Integer(100, 1000),  # Keep this as Integer for BayesSearchCV
    "subsample": Real(0.7, 1.0),
    "colsample_bytree": Real(0.7, 1.0)
}

opt = BayesSearchCV(xgb, param_space, n_iter=30, scoring="neg_log_loss", cv=3, random_state=42)
opt.fit(X_train_scaled, y_train)

print(f"🔹 Optimized XGBoost Log Loss (BayesSearchCV): {-opt.best_score_}")
print(f"🔹 Best XGBoost Parameters from BayesSearchCV: {opt.best_params_}")

# 🏀 Randomized Search Optimization (Fixed `n_estimators`)
random_param_grid = {
    'n_estimators': [100, 250, 500, 750, 1000],  # Explicit integer values
    'learning_rate': [0.005, 0.01, 0.02],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0]
}

search = RandomizedSearchCV(xgb, param_distributions=random_param_grid, 
                            n_iter=15, scoring="neg_log_loss", cv=3, verbose=2, 
                            n_jobs=-1, random_state=42)

search.fit(X_train_scaled, y_train)

print(f"🔹 Optimized XGBoost Log Loss (RandomizedSearchCV): {-search.best_score_}")
print(f"🔹 Best XGBoost Parameters from RandomizedSearchCV: {search.best_params_}")

🔹 Stacked Model Log Loss: 0.4154238283793958
🔹 Optimized XGBoost Log Loss (BayesSearchCV): 0.6952508346859395
🔹 Best XGBoost Parameters from BayesSearchCV: OrderedDict([('colsample_bytree', 0.7), ('learning_rate', 0.01), ('max_depth', 3), ('n_estimators', 100), ('subsample', 0.996093287631074)])
Fitting 3 folds for each of 15 candidates, totalling 45 fits
🔹 Optimized XGBoost Log Loss (RandomizedSearchCV): 0.6986199819011332
🔹 Best XGBoost Parameters from RandomizedSearchCV: {'subsample': 0.8, 'n_estimators': 500, 'max_depth': 3, 'learning_rate': 0.005, 'colsample_bytree': 0.8}


: 